### SAX alphabet size & PAA word size (FARSEEING CV)

In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.simplefilter(action='ignore')

from src.costream.data import group_files_by_subject
from src.costream.evaluation import run_subject_cv, ModelSpec
import numpy as np
import pandas as pd
from src.fall_lm import FallLM
from src import utils

FALL_DIR = "data/farseeing/fall"
FREQ = 100
WINDOW_SEC = 3

In [2]:
data_desc = pd.read_excel("data/farseeing/description.xlsx", engine='openpyxl',
                          usecols=['Randomnumber', 'Sensor_location', 'Sample_rate_Hz'])
l5 = data_desc[data_desc['Sensor_location'] == 'L5']
l5 = l5[l5['Sample_rate_Hz'] == FREQ]
l5_subjects = set(l5['Randomnumber'].str.split('-').str[0])

fall_subject_map = group_files_by_subject(FALL_DIR, id_extractor=utils.fall_id_extractor)
fall_subject_map = {s: fps for s, fps in fall_subject_map.items() if s in l5_subjects}
print(f"Found {len(fall_subject_map)} fall subjects")

Found 40 fall subjects


In [3]:
subject_df_map = {s: utils.load_fall_subject(fall_subject_map[s]) for s in fall_subject_map.keys()}

In [ ]:
BASE_KW = dict(vectorizer_type="tfidf", alpha=1.5, use_diff=False, impact_mode="category", random_state=42)
alphabet_specs = [
    ModelSpec(f"n_bins={n}", FallLM(n_bins=n, word_size=3, ngram_range=(4, 5), **BASE_KW))
    for n in range(3, 11)
]
wordsize_specs = [
    ModelSpec(f"word_size={w}", FallLM(n_bins=5, word_size=w, ngram_range=(4, 5), **BASE_KW))
    for w in range(3, 31, 3)
]

specs = alphabet_specs + wordsize_specs
len(specs)

18

In [5]:
import os
os.makedirs('output/results', exist_ok=True)

cv_res = run_subject_cv(
    subject_map=subject_df_map,
    model_specs=specs,
    feature_cols=['mag'],
    window_size=WINDOW_SEC,
    cv=5,
    step=1.0,
    spacing="multiphase",
    verbose=True,
    use_post_event_data=False,
    tolerance=[3, 20],
)
cv_res.to_csv('output/results/cv_results_ablation.csv', index=False)


=== Fold 1/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (1627, 300)



=== Fold 2/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2563, 300)



=== Fold 3/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2380, 300)



=== Fold 4/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2647, 300)



=== Fold 5/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2755, 300)


In [6]:
metrics = ["f1-score", "precision", "recall", "false alarm rate"]

def agg_table(df, models, key, key_name):
    sub = df[df["model"].isin(models)].copy()
    sub[key_name] = sub["model"].map(key)
    g = sub.groupby(key_name)
    out = g[metrics].mean().round(3)
    for m in metrics:
        out[m + "_std"] = g[m].std().round(3)
    return out.reset_index().sort_values(key_name)

In [7]:
alphabet_models = [s.name for s in alphabet_specs]
alphabet_key = {s.name: int(s.name.split("=")[1]) for s in alphabet_specs}
alphabet_table = agg_table(cv_res, alphabet_models, alphabet_key, "n_bins")
alphabet_table.to_csv("output/results/ablation_alphabet.csv", index=False)
alphabet_table.to_latex("output/results/ablation_alphabet.tex", index=False, float_format="%.3f")
alphabet_table

,n_bins,f1-score,precision,recall,false alarm rate,f1-score_std,precision_std,recall_std,false alarm rate_std
0,3,0.641,0.512,0.882,0.296,0.103,0.122,0.083,0.148
1,4,0.516,0.420,0.767,0.477,0.172,0.217,0.085,0.358
2,5,0.662,0.542,0.880,0.271,0.115,0.137,0.088,0.168
3,6,0.647,0.540,0.838,0.260,0.105,0.150,0.042,0.144
4,7,0.634,0.524,0.830,0.281,0.122,0.144,0.075,0.194
5,8,0.623,0.506,0.842,0.296,0.105,0.146,0.039,0.149
6,9,0.611,0.488,0.853,0.348,0.161,0.169,0.076,0.240
7,10,0.645,0.526,0.852,0.272,0.101,0.129,0.040,0.142


In [8]:
wordsize_models = [s.name for s in wordsize_specs]
wordsize_key = {s.name: int(s.name.split("=")[1]) for s in wordsize_specs}
wordsize_table = agg_table(cv_res, wordsize_models, wordsize_key, "word_size")
wordsize_table.to_csv("output/results/ablation_wordsize.csv", index=False)
wordsize_table.to_latex("output/results/ablation_wordsize.tex", index=False, float_format="%.3f")
wordsize_table

,word_size,f1-score,precision,recall,false alarm rate,f1-score_std,precision_std,recall_std,false alarm rate_std
0,3,0.662,0.542,0.880,0.271,0.115,0.137,0.088,0.168
1,6,0.580,0.453,0.835,0.367,0.132,0.135,0.103,0.197
2,9,0.594,0.467,0.838,0.328,0.089,0.104,0.100,0.123
3,12,0.576,0.452,0.820,0.345,0.114,0.130,0.071,0.127
4,15,0.570,0.441,0.809,0.335,0.103,0.089,0.130,0.072
5,18,0.578,0.444,0.835,0.341,0.083,0.074,0.100,0.063
6,21,0.535,0.398,0.826,0.416,0.089,0.088,0.070,0.101
7,24,0.511,0.375,0.813,0.447,0.068,0.066,0.069,0.093
8,27,0.498,0.361,0.813,0.475,0.059,0.059,0.043,0.101
9,30,0.460,0.331,0.764,0.502,0.044,0.039,0.099,0.090


### SAX alphabet size ablation on FallAllD (subject-wise 5-fold CV)

In [9]:
from src.fallalld import load_fallalld, adapter_fallalld_to_costream

FALLALLD_ORIG_FREQ = 238

fallalld_df = load_fallalld()
fallalld_subject_map_raw = adapter_fallalld_to_costream(fallalld_df)

fallalld_subject_map = {
    subj: [utils.resample_df(rec, ['mag'], FALLALLD_ORIG_FREQ, FREQ) for rec in recs]
    for subj, recs in fallalld_subject_map_raw.items()
}
print(f"{len(fallalld_subject_map)} FallAllD subjects")

Adapting DataFrame to Costream format...



  0%|          | 0/1745 [00:00<?, ?it/s]


 45%|████▍     | 777/1745 [00:00<00:00, 7758.21it/s]


 90%|████████▉ | 1566/1745 [00:00<00:00, 7834.81it/s]


100%|██████████| 1745/1745 [00:00<00:00, 7776.32it/s]

14 FallAllD subjects


In [10]:
cv_res_fallalld = run_subject_cv(
    subject_map=fallalld_subject_map,
    model_specs=alphabet_specs,
    feature_cols=['mag'],
    window_size=WINDOW_SEC,
    cv=5,
    step=1.0,
    spacing="multiphase",
    verbose=True,
    use_post_event_data=False,
    tolerance=[3, 20],
)
cv_res_fallalld.to_csv('output/results/cv_results_ablation_fallalld.csv', index=False)


=== Fold 1/5 | Train Subjects: 11 | Test Subjects: 3 ===


  Segmented Train Data: (6829, 300)



=== Fold 2/5 | Train Subjects: 11 | Test Subjects: 3 ===


  Segmented Train Data: (5391, 300)



=== Fold 3/5 | Train Subjects: 11 | Test Subjects: 3 ===


  Segmented Train Data: (5795, 300)



=== Fold 4/5 | Train Subjects: 11 | Test Subjects: 3 ===


  Segmented Train Data: (4860, 300)



=== Fold 5/5 | Train Subjects: 12 | Test Subjects: 2 ===


  Segmented Train Data: (5997, 300)


In [11]:
alphabet_table_fallalld = agg_table(cv_res_fallalld, alphabet_models, alphabet_key, "n_bins")
alphabet_table_fallalld.to_csv("output/results/ablation_alphabet_fallalld.csv", index=False)
alphabet_table_fallalld.to_latex("output/results/ablation_alphabet_fallalld.tex", index=False, float_format="%.3f")
alphabet_table_fallalld

,n_bins,f1-score,precision,recall,false alarm rate,f1-score_std,precision_std,recall_std,false alarm rate_std
0,3,0.745,0.607,0.971,2.918,0.047,0.061,0.028,0.733
1,4,0.753,0.616,0.973,2.824,0.046,0.061,0.030,0.734
2,5,0.746,0.606,0.976,2.953,0.047,0.062,0.026,0.769
3,6,0.751,0.615,0.969,2.824,0.048,0.062,0.030,0.734
4,7,0.727,0.667,0.833,2.131,0.066,0.058,0.189,1.317
5,8,0.750,0.615,0.968,2.824,0.042,0.059,0.027,0.734
6,9,0.740,0.671,0.826,1.892,0.071,0.081,0.059,0.580
7,10,0.741,0.613,0.943,2.779,0.050,0.064,0.036,0.751
